In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
project_dir = Path.cwd().parent
data_dir = project_dir / "data" / "processed"

# Load datasets
freq = pd.read_csv(data_dir / "freMTPL2freq.csv")
sev = pd.read_csv(data_dir / "freMTPL2sev.csv")

print("Frequency dataset:", freq.shape)
print("Severity dataset:", sev.shape)

Frequency dataset: (677991, 12)
Severity dataset: (26444, 2)


## 1. Data Structure and Completeness

The first stage of the data quality audit examines variable types and missing values in both the frequency and severity datasets.


In [3]:
print("FREQUENCY DATASET")
print("=" * 50)
freq.info()

print("\nMissing values:")
print(freq.isna().sum())

print("\nSEVERITY DATASET")
print("=" * 50)
sev.info()

print("\nMissing values:")
print(sev.isna().sum())

FREQUENCY DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 677991 entries, 0 to 677990
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDpol       677991 non-null  float64
 1   ClaimNb     677991 non-null  int64  
 2   Exposure    677991 non-null  float64
 3   VehPower    677991 non-null  int64  
 4   VehAge      677991 non-null  int64  
 5   DrivAge     677991 non-null  int64  
 6   BonusMalus  677991 non-null  int64  
 7   VehBrand    677991 non-null  object 
 8   VehGas      677991 non-null  object 
 9   Area        677991 non-null  object 
 10  Density     677991 non-null  int64  
 11  Region      677991 non-null  object 
dtypes: float64(2), int64(6), object(4)
memory usage: 62.1+ MB

Missing values:
IDpol         0
ClaimNb       0
Exposure      0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Area          0
Density       0
Region        0
dtype: int6

In [4]:
print("Frequency Missing Values")
print(freq.isnull().sum().sort_values(ascending=False))

print("\nSeverity Missing Values")
print(sev.isnull().sum().sort_values(ascending=False))


Frequency Missing Values
IDpol         0
ClaimNb       0
Exposure      0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Area          0
Density       0
Region        0
dtype: int64

Severity Missing Values
IDpol          0
ClaimAmount    0
dtype: int64


In [5]:
print("Frequency duplicates:", freq.duplicated().sum())
print("Severity duplicates:", sev.duplicated().sum())

Frequency duplicates: 0
Severity duplicates: 235


In [6]:
# Inspect exact duplicate rows in the severity dataset

sev_duplicates = sev[sev.duplicated(keep=False)].sort_values(
    ["IDpol", "ClaimAmount"]
)

print("Rows involved in exact duplicate groups:", len(sev_duplicates))
print("Number of duplicate rows beyond first occurrence:", sev.duplicated().sum())
print("Number of unique duplicated IDpol values:", sev_duplicates["IDpol"].nunique())

sev_duplicates.head(30)

Rows involved in exact duplicate groups: 464
Number of duplicate rows beyond first occurrence: 235
Number of unique duplicated IDpol values: 227


,IDpol,ClaimAmount
5022,18698,1128.12
6355,18698,1128.12
4514,19887,1128.12
4521,19887,1128.12
6318,24992,1128.12
6322,24992,1128.12
5862,25763,1128.12
5866,25763,1128.12
24094,29567,1128.12
24096,29567,1128.12


In [7]:
# Validate severity claim counts against ClaimNb in frequency data

severity_counts = (
    sev.groupby("IDpol")
       .size()
       .rename("SeverityRecordCount")
       .reset_index()
)

claim_validation = freq[["IDpol", "ClaimNb"]].merge(
    severity_counts,
    on="IDpol",
    how="left"
)

claim_validation["SeverityRecordCount"] = (
    claim_validation["SeverityRecordCount"]
    .fillna(0)
    .astype(int)
)

claim_validation["Difference"] = (
    claim_validation["ClaimNb"] -
    claim_validation["SeverityRecordCount"]
)

print("Policies checked:", len(claim_validation))
print("Policies with matching claim counts:",
      (claim_validation["Difference"] == 0).sum())
print("Policies with different claim counts:",
      (claim_validation["Difference"] != 0).sum())

print("\nDifference summary:")
print(claim_validation["Difference"].value_counts().sort_index())

Policies checked: 677991
Policies with matching claim counts: 677991
Policies with different claim counts: 0

Difference summary:
Difference
0    677991
Name: count, dtype: int64


### Data Quality Finding: Apparent Duplicate Severity Records

The severity dataset contains 235 rows identified as exact duplicates based on `IDpol` and `ClaimAmount`. These records were investigated rather than automatically removed.

A reconciliation was performed between the number of claim records in the severity dataset and the policy-level `ClaimNb` variable in the frequency dataset.

**Validation results:**
- Policies checked: 677,991
- Policies with matching claim counts: 677,991
- Policies with mismatched claim counts: 0

The severity records therefore reconcile exactly with the reported policy-level claim counts. The apparent duplicates represent valid claim observations, including cases where a policy has multiple claims with the same recorded claim amount.

**Decision:** All severity records are retained. Removing the 235 apparent duplicates would incorrectly reduce the number of claims and distort aggregate loss and severity calculations.

## 2. Numerical Variable Validation

This section examines the ranges and distributions of key numerical variables to identify impossible, unusual, or potentially influential observations before modelling.

In [8]:
# Summary statistics for key frequency variables

numeric_cols = [
    "ClaimNb",
    "Exposure",
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "Density"
]

freq[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
ClaimNb,677991.0,0.039003,0.207169,0.000000,0.00,0.00,0.00,16.00
Exposure,677991.0,0.528743,0.364441,0.002732,0.18,0.49,0.99,2.01
VehPower,677991.0,6.454649,2.050914,4.000000,5.00,6.00,7.00,15.00
VehAge,677991.0,7.044244,5.666226,0.000000,2.00,6.00,11.00,100.00
DrivAge,677991.0,45.499092,14.137449,18.000000,34.00,44.00,55.00,100.00
BonusMalus,677991.0,59.761559,15.636713,50.000000,50.00,50.00,64.00,230.00
Density,677991.0,1792.406856,3958.574057,1.000000,92.00,393.00,1658.00,27000.00


In [9]:
# Summary statistics for claim severity

sev["ClaimAmount"].describe()

count    2.644400e+04
mean     2.265513e+03
std      2.937103e+04
min      1.000000e+00
25%      6.859925e+02
50%      1.172000e+03
75%      1.212385e+03
max      4.075401e+06
Name: ClaimAmount, dtype: float64

In [10]:
# Examine the upper tail of the claim severity distribution

severity_percentiles = sev["ClaimAmount"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999]
)

print("Claim severity percentiles:")
print(severity_percentiles)

print("\nLargest 20 claims:")
print(
    sev.nlargest(20, "ClaimAmount")
       [["IDpol", "ClaimAmount"]]
       .to_string(index=False)
)

print("\nClaims above €100,000:",
      (sev["ClaimAmount"] > 100000).sum())

print("Claims above €500,000:",
      (sev["ClaimAmount"] > 500000).sum())

print("Claims above €1,000,000:",
      (sev["ClaimAmount"] > 1000000).sum())

Claim severity percentiles:
0.900      2767.60400
0.950      4765.09350
0.990     16451.22400
0.995     34376.96435
0.999    152223.24082
Name: ClaimAmount, dtype: float64

Largest 20 claims:
  IDpol  ClaimAmount
1120377   4075400.56
 110846   1403057.40
2141337   1301172.60
3122016    774411.50
2008127    390742.27
3025890    369131.88
1117644    307096.42
 158309    301635.49
3075820    287423.00
3150210    281897.49
4169019    276643.00
1096763    255013.03
2206511    241171.16
4159792    211204.00
3103319    209695.87
3059946    208472.60
1067220    205432.05
2070262    205132.00
2202813    203172.00
5033465    200885.00

Claims above €100,000: 41
Claims above €500,000: 4
Claims above €1,000,000: 3


### Claim Severity Tail Assessment

The claim severity distribution is strongly right-skewed and contains a small number of very large losses.

Key findings:

- Median claim amount: approximately €1,172
- 95th percentile: approximately €4,765
- 99th percentile: approximately €16,451
- 99.5th percentile: approximately €34,377
- 99.9th percentile: approximately €152,223
- Maximum observed claim: approximately €4.08 million
- 41 claims exceed €100,000
- 4 claims exceed €500,000
- 3 claims exceed €1 million

The extreme observations are retained. In a capital and risk modelling context, large claims are particularly important because they contribute to the tail of the aggregate loss distribution and may materially influence high-confidence risk measures such as VaR and TVaR.

No arbitrary outlier removal or capping is applied at the data-quality stage. The suitability of alternative severity distributions for modelling the observed tail will be assessed during the severity-modelling phase.

In [11]:
# Investigate unusual numerical values in frequency data

checks = {
    "ClaimNb > 4": (freq["ClaimNb"] > 4).sum(),
    "Exposure > 1": (freq["Exposure"] > 1).sum(),
    "VehAge >= 50": (freq["VehAge"] >= 50).sum(),
    "DrivAge >= 90": (freq["DrivAge"] >= 90).sum(),
    "BonusMalus > 150": (freq["BonusMalus"] > 150).sum()
}

for check, count in checks.items():
    print(f"{check}: {count:,}")

ClaimNb > 4: 8
Exposure > 1: 1,224
VehAge >= 50: 105
DrivAge >= 90: 568
BonusMalus > 150: 209


### Frequency Variable Range Assessment

Several uncommon values were identified during numerical validation:

- 8 policies report more than 4 claims.
- 1,224 observations have exposure greater than 1.
- 105 vehicles have recorded ages of at least 50 years.
- 568 drivers have recorded ages of at least 90 years.
- 209 observations have Bonus-Malus values above 150.

These observations represent a small proportion of the overall portfolio and are not automatically treated as errors or removed. Extreme values may reflect legitimate policy characteristics, dataset coding conventions, or unusual risks.

No adjustments are made at this stage. Variables requiring modelling constraints or transformations will be addressed during the relevant modelling phase, with the original observations preserved.

## 3. Categorical Variable Validation

Categorical variables are reviewed to identify unexpected categories, inconsistent coding, blank values, and highly sparse groups that may affect subsequent modelling.

In [12]:
# Review categorical variables

categorical_cols = ["VehBrand", "VehGas", "Area", "Region"]

for col in categorical_cols:
    print(f"\n{'=' * 50}")
    print(f"{col}")
    print(f"{'=' * 50}")
    print("Number of categories:", freq[col].nunique(dropna=False))
    print("\nCategory counts:")
    print(freq[col].value_counts(dropna=False).sort_index())


VehBrand
Number of categories: 11

Category counts:
VehBrand
B1     162730
B10     17707
B11     13585
B12    166022
B13     12177
B14      4047
B2     159854
B3      53393
B4      25179
B5      34752
B6      28545
Name: count, dtype: int64

VehGas
Number of categories: 2

Category counts:
VehGas
Diesel     332128
Regular    345863
Name: count, dtype: int64

Area
Number of categories: 6

Category counts:
Area
A    103952
B     75457
C    191874
D    151592
E    137163
F     17953
Name: count, dtype: int64

Region
Number of categories: 22

Category counts:
Region
Alsace                           2200
Aquitaine                       31329
Auvergne                         5287
Basse-Normandie                 10892
Bourgogne                       10491
Bretagne                        42120
Centre                         160595
Champagne-Ardenne                3025
Corse                            4516
Franche-Comte                    1326
Haute-Normandie                  8784
Ile-de-Franc

### Categorical Variable Assessment

The categorical variables were reviewed for missing categories, inconsistent labels, unexpected coding and sparse groups.

**Results:**
- `VehBrand`: 11 categories
- `VehGas`: 2 categories
- `Area`: 6 categories
- `Region`: 22 categories
- No missing or blank categorical values were identified.
- Category labels are internally consistent.

The portfolio is unevenly distributed across some categories. For example, certain vehicle-brand groups and geographic regions contain substantially fewer observations than others. However, no categories are removed or combined during the data-quality stage.

**Decision:** All original categorical groups are retained. Any category grouping required for statistical stability will be evaluated during model development using exposure, claim experience and modelling evidence rather than observation count alone.

## 4. Exposure and Claim Consistency

This section tests key actuarial relationships between policy exposure and observed claim counts before frequency modelling.

In [13]:
# Exposure and claim consistency checks

print("Exposure <= 0:",
      (freq["Exposure"] <= 0).sum())

print("Claims with Exposure <= 0:",
      ((freq["ClaimNb"] > 0) & (freq["Exposure"] <= 0)).sum())

print("Policies with claims:",
      (freq["ClaimNb"] > 0).sum())

print("Policies with no claims:",
      (freq["ClaimNb"] == 0).sum())

print("Total exposure:",
      round(freq["Exposure"].sum(), 2))

print("Total reported claims:",
      freq["ClaimNb"].sum())

print("Overall observed claim frequency:",
      round(freq["ClaimNb"].sum() / freq["Exposure"].sum(), 6))

Exposure <= 0: 0
Claims with Exposure <= 0: 0
Policies with claims: 24944
Policies with no claims: 653047
Total exposure: 358482.84
Total reported claims: 26444
Overall observed claim frequency: 0.073766


### Exposure and Claim Consistency Assessment

The relationship between policy exposure and reported claims was validated before frequency modelling.

**Results:**
- No policies have zero or negative exposure.
- No claims are associated with zero or negative exposure.
- 24,944 policies reported at least one claim.
- 653,047 policies reported no claims.
- Total portfolio exposure is 358,482.84 policy-years.
- Total reported claims are 26,444.
- Overall observed claim frequency is approximately 0.07377 claims per policy-year, equivalent to approximately 7.38 claims per 100 policy-years.

The difference between the number of claim-bearing policies (24,944) and total claims (26,444) reflects policies experiencing multiple claims and is consistent with the earlier reconciliation between the frequency and severity datasets.

**Decision:** No exposure or claim-count records require removal on the basis of these consistency checks.

### Exposure Validation

The CASdatasets documentation defines `Exposure` as the period of exposure for a policy, measured in years. The freMTPL2 portfolio contains 677,991 policies observed mostly, but not necessarily exclusively, over one year.

The data contain 1,224 observations with exposure greater than one year, with a maximum observed exposure of 2.01 years. These observations are therefore not treated as errors solely because they exceed one year.

**Decision:** Original exposure values are retained. Exposure will be incorporated explicitly into the claim-frequency model rather than arbitrarily capped at one year.

### Data Quality Audit Conclusion

The frequency and severity datasets passed the principal integrity checks required before actuarial modelling:

- Dataset dimensions reconcile with the original CASdatasets source files.
- No missing values were identified.
- Policy-level claim counts reconcile exactly with the 26,444 individual severity records.
- Apparent duplicate severity records are retained following reconciliation.
- Extreme claim amounts are retained because tail losses are material to capital modelling.
- Numerical and categorical variables have been reviewed for unusual values.
- All policies have positive exposure.
- Exposure values above one year are retained based on the documented definition of exposure.

No arbitrary deletion, winsorisation, capping or imputation has been applied during the data-quality stage. Any transformations required for statistical modelling will be evaluated and documented during model development.